# 05 – Label Validation & Machine Learning

**NASA Challenge – Inspiration4**

Notebook này thực hiện:
1. Kiểm tra tính hợp lệ và phân bố label `Preflight` / `Postflight`.
2. Kiểm tra quy ước thời điểm `L-` và `R+`, phát hiện giá trị thời gian bất thường.
3. Vẽ biểu đồ phân bố label và đưa ra nhận xét dựa trên kết quả thực tế.
4. Huấn luyện và so sánh Dummy baseline cùng 3 mô hình học máy.
5. Đánh giá khả năng tổng quát hóa bằng **Leave-One-Astronaut-Out (LOAO)**.

**Quan trọng:** CBC + Serum và Plasma Proteomics được đánh giá riêng. Không ghép hai bảng theo tên phi hành gia hoặc Phase vì chưa xác nhận được chúng là cùng thời điểm lấy mẫu. Với chỉ 4 phi hành gia và 21–28 mẫu, kết quả chỉ mang tính thăm dò; không thể cam kết metric cao hoặc khẳng định mô hình tổng quát hóa tốt cho phi hành đoàn khác.

## 1. Import thư viện và tìm thư mục dự án

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

def find_project_root():
    current = Path.cwd().resolve()
    for folder in [current, *current.parents]:
        if (folder / "processed_data").is_dir():
            return folder
    raise FileNotFoundError("Không tìm thấy thư mục processed_data.")

BASE_DIR = find_project_root()
DATA_DIR = BASE_DIR / "processed_data"
FIG_DIR = BASE_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

print("Project:", BASE_DIR)

FileNotFoundError: Không tìm thấy thư mục processed_data.

## 2. Load dữ liệu và xác định label

`Phase` là biến mục tiêu. Notebook đọc các bảng tích hợp **chưa lọc đặc trưng** từ notebook 04. Các cột định danh/thời gian và metadata không được dùng làm predictors. Việc loại cột hằng, cột thiếu nhiều và chọn đặc trưng được fit riêng trên training fold trong LOAO để tránh data leakage.

In [ ]:
# Dùng bảng tích hợp CHƯA lọc đặc trưng.
# Việc lọc đặc trưng toàn bộ dữ liệu trước LOAO có thể làm lộ thông tin từ
# phi hành gia đang được giữ lại làm test (data leakage).
FILES = {
    "CBC + Serum": DATA_DIR / "CBC_Serum_Integrated.csv",
    "Plasma Proteomics": DATA_DIR / "Plasma_Proteomics_Integrated.csv"
}

datasets = {}
for name, path in FILES.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Thiếu file dữ liệu tích hợp: {path}. "
            "Hãy chạy notebook 04_Data_Integration.ipynb trước."
        )
    df = pd.read_csv(path)
    required = {"Sample_ID", "Astronaut", "Phase"}
    if not required.issubset(df.columns):
        raise ValueError(f"{name}: thiếu cột bắt buộc {required - set(df.columns)}")
    if df["Sample_ID"].duplicated().any():
        raise ValueError(f"{name}: Sample_ID bị trùng; cần xử lý trước khi train.")
    if df["Astronaut"].isna().any() or df["Phase"].isna().any():
        raise ValueError(f"{name}: Astronaut hoặc Phase có giá trị thiếu.")
    datasets[name] = df
    print(f"{name}: {df.shape[0]} samples, {df.shape[1]} columns")
    print("  Phase:", df["Phase"].value_counts().to_dict())
    print("  Astronauts:", sorted(df["Astronaut"].astype(str).unique()))

## 3. Kiểm tra tính hợp lệ của label và thời điểm

- CBC + Serum: kiểm tra `L-...` phải tương ứng `Preflight`; `R+...` phải tương ứng `Postflight`.
- Plasma: đối chiếu `Phase` với hậu tố `_Pre` / `_Post` trong `Timepoint`.
- `L-92` không phải ngày lịch âm. Đây là mốc tương đối 92 ngày trước phóng; `R+194` là 194 ngày sau khi trở về.
- Nếu có giá trị số âm trong một cột ngày lịch thực, cần kiểm tra riêng theo định nghĩa cột, không tự động đổi thành số dương.

In [ ]:
label_checks = []

cbc = datasets["CBC + Serum"].copy()
cbc["Expected_Phase_from_Timepoint"] = np.where(
    cbc["Timepoint"].astype(str).str.startswith("L-"), "Preflight",
    np.where(cbc["Timepoint"].astype(str).str.startswith("R+"), "Postflight", "UNKNOWN")
)
cbc["Label_Match"] = cbc["Phase"].eq(cbc["Expected_Phase_from_Timepoint"])
label_checks.append({
    "Dataset": "CBC + Serum",
    "Rows": len(cbc),
    "Missing_Label": int(cbc["Phase"].isna().sum()),
    "Unique_Labels": ", ".join(sorted(cbc["Phase"].dropna().astype(str).unique())),
    "Invalid_or_Mismatched": int((~cbc["Label_Match"]).sum()),
    "Unique_Astronauts": cbc["Astronaut"].nunique()
})

plasma = datasets["Plasma Proteomics"].copy()
if "Timepoint" in plasma.columns:
    tp = plasma["Timepoint"].astype(str)
    expected = np.where(tp.str.endswith("_Pre"), "Preflight",
               np.where(tp.str.endswith("_Post"), "Postflight", "UNKNOWN"))
    plasma["Expected_Phase_from_Timepoint"] = expected
    plasma["Label_Match"] = plasma["Phase"].eq(plasma["Expected_Phase_from_Timepoint"])
    mismatch = int((~plasma["Label_Match"]).sum())
else:
    mismatch = np.nan

label_checks.append({
    "Dataset": "Plasma Proteomics",
    "Rows": len(plasma),
    "Missing_Label": int(plasma["Phase"].isna().sum()),
    "Unique_Labels": ", ".join(sorted(plasma["Phase"].dropna().astype(str).unique())),
    "Invalid_or_Mismatched": mismatch,
    "Unique_Astronauts": plasma["Astronaut"].nunique()
})

label_validation = pd.DataFrame(label_checks)
display(label_validation)

for name, df in datasets.items():
    print(f"\n{name}: duplicated Sample_ID =", df["Sample_ID"].duplicated().sum())
    print("Unknown labels =", sorted(set(df["Phase"].dropna()) - {"Preflight", "Postflight"}))

In [ ]:
# 3b. Kiểm tra tính hợp lệ của Mission_Day (CBC + Serum)
# Mission_Day = số ngày tương đối suy ra từ Timepoint: "L-92" -> -92 (92 ngày TRƯỚC phóng),
# "R+1" -> +1 (1 ngày SAU khi về). Giá trị âm là hợp lệ theo quy ước này, không phải lỗi.
# Lưu ý: L- tính từ lúc phóng còn R+ tính từ lúc về, và không có ngày 0, nên hiệu giữa -3 và +1
# KHÔNG bằng số ngày thực tế (chưa tính thời gian bay). Chỉ dùng dấu của Mission_Day để gán nhãn;
# không nên coi đây là một trục thời gian liên tục khi vẽ xu hướng.
EXPECTED_DAYS = [-92, -44, -3, 1, 45, 82, 194]

md_df = datasets["CBC + Serum"][["Sample_ID", "Astronaut", "Timepoint", "Phase"]].copy()
parsed = md_df["Timepoint"].astype(str).str.extract(r"^(?P<origin>[LR])(?P<sign>[+-])(?P<days>\d+)$")
md_df["Origin"] = parsed["origin"]
md_df["Mission_Day"] = pd.to_numeric(parsed["sign"] + parsed["days"], errors="coerce")

known = md_df["Mission_Day"].notna()
expected_phase = np.where(md_df["Mission_Day"] < 0, "Preflight", "Postflight")
unexpected_vals = sorted(set(md_df.loc[known, "Mission_Day"]) - set(EXPECTED_DAYS))
bad_origin = ((md_df["Origin"] == "L") & (md_df["Mission_Day"] > 0)) | \
             ((md_df["Origin"] == "R") & (md_df["Mission_Day"] < 0))
bad_phase = known & (md_df["Phase"] != expected_phase)
incomplete = [
    a for a, g in md_df.groupby("Astronaut")
    if sorted(g["Mission_Day"].dropna().astype(int)) != sorted(EXPECTED_DAYS)
]

checks = [
    ("Timepoint parse được thành Mission_Day", int((~known).sum()), "số dòng không parse được"),
    ("Giá trị nằm trong danh sách mốc cho phép", len(unexpected_vals), f"giá trị lạ: {unexpected_vals}"),
    ("Không có Mission_Day = 0", int((md_df["Mission_Day"] == 0).sum()), "quy ước L-/R+ không có ngày 0"),
    ("Không có Mission_Day = -31", int((md_df["Mission_Day"] == -31).sum()),
     "-31 từng xuất hiện trên trục X của notebook 575 nhưng không phải mốc hợp lệ"),
    ("Tiền tố L-/R+ khớp dấu của Mission_Day", int(bad_origin.sum()), "L phải âm, R phải dương"),
    ("Dấu của Mission_Day khớp Phase", int(bad_phase.sum()), "âm = Preflight, dương = Postflight"),
    ("Mỗi phi hành gia có đủ 7 mốc, không trùng", len(incomplete), f"phi hành gia thiếu/trùng: {incomplete}"),
]
mission_day_report = pd.DataFrame([
    {"Check": c, "Failed": n, "Status": "PASS" if n == 0 else "FAIL", "Detail": d} for c, n, d in checks
])
display(mission_day_report)

day_summary = (md_df.groupby(["Mission_Day", "Phase"]).size()
               .reset_index(name="Samples").sort_values("Mission_Day"))
display(day_summary)

print("Kết luận:", "Mission_Day hợp lệ; giá trị âm = ngày trước phóng, khớp nhãn Preflight."
      if (mission_day_report["Status"] == "PASS").all()
      else "Có kiểm tra FAIL, cần rà lại dữ liệu trước khi dùng Mission_Day.")
mission_day_report.to_csv(DATA_DIR / "Mission_Day_Validation_Report.csv", index=False)

## 4. Phân bố label và nhận xét

Tỷ lệ lớp được tính trực tiếp từ dữ liệu hiện tại. Nếu tỷ lệ lớp thấp hơn 20% hoặc cao hơn 80%, cần xem xét mất cân bằng đáng kể. Với dữ liệu nhỏ, vẫn phải báo cáo cả số lượng và tỷ lệ, không chỉ nhìn biểu đồ.

In [ ]:
class_rows = []
for name, df in datasets.items():
    counts = df["Phase"].value_counts(dropna=False)
    total = len(df)
    for label, count in counts.items():
        class_rows.append({
            "Dataset": name, "Phase": label,
            "Samples": int(count), "Percentage": 100 * count / total
        })

class_distribution = pd.DataFrame(class_rows)
display(class_distribution)

for name, df in datasets.items():
    counts = df["Phase"].value_counts()
    ratio = counts.min() / counts.max() if len(counts) == 2 else np.nan
    print(f"\n{name}")
    print(counts.to_string())
    if pd.notna(ratio) and ratio >= 0.5:
        print("Nhận xét: hai lớp tương đối cân bằng; chưa thấy mất cân bằng nghiêm trọng.")
    elif pd.notna(ratio):
        print("Nhận xét: phân bố lớp lệch; cần ưu tiên Balanced Accuracy, F1 và class_weight.")
    else:
        print("Nhận xét: label không đủ hai lớp; không thể huấn luyện phân loại nhị phân.")

In [ ]:
for name, df in datasets.items():
    counts = df["Phase"].value_counts().reindex(["Preflight", "Postflight"], fill_value=0)
    ax = counts.plot(kind="bar", figsize=(6, 4), rot=0)
    ax.set_title(f"Label distribution – {name}")
    ax.set_xlabel("Phase")
    ax.set_ylabel("Number of samples")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"label_distribution_{name.lower().replace(' ', '_').replace('+','and')}.png", dpi=150)
    plt.show()

## 5. Kiểm tra phân bố theo phi hành gia

LOAO chỉ có ý nghĩa nếu mỗi fold có thể huấn luyện với hai lớp và tập kiểm tra cũng có đủ hai lớp. Bảng chéo dưới đây cho biết mỗi phi hành gia đóng góp bao nhiêu mẫu cho từng label.

In [ ]:
for name, df in datasets.items():
    print("\n", name)
    display(pd.crosstab(df["Astronaut"], df["Phase"], margins=True))

## 6. Chuẩn bị mô hình và chiến lược đánh giá

So sánh:
- Dummy Classifier: baseline tối thiểu.
- Logistic Regression: mô hình tuyến tính, có `class_weight='balanced'`.
- SVM RBF: mô hình phi tuyến.
- Random Forest: mô hình cây tổ hợp.

**Chống data leakage:** feature filtering (loại biến hằng, biến thiếu trên 50%, biến min/max/range), imputation, chọn tối đa 10 đặc trưng và scaling đều nằm trong Pipeline. Mỗi bước chỉ được fit trên training fold của từng lượt LOAO. Không sử dụng `Mission_Day`, `Astronaut`, `Phase`, `Timepoint` hay các cột định danh làm predictors. Siêu tham số được cố định trước, không tuning dựa trên test fold.

In [ ]:
TARGET_MAP = {"Preflight": 0, "Postflight": 1}

# Các cột metadata, kể cả cột số, tuyệt đối không đưa vào mô hình.
METADATA_COLUMNS = {
    "Sample_ID", "Sample Name", "Sample_Base", "Sample_name",
    "Astronaut", "Phase", "Timepoint", "Mission_Day", "MissionDay",
    "Origin", "PreVpost", "ImmediateVLongTerm", "Mission_Date",
    "MissionDate", "Date", "Collection_Date"
}

def make_feature_frame(df):
    """Chỉ lấy predictors số; loại metadata/ID kể cả khi chúng có dạng số."""
    metadata_lower = {str(c).lower() for c in METADATA_COLUMNS}
    drop_cols = [c for c in df.columns if str(c).lower() in metadata_lower]
    X = df.drop(columns=drop_cols, errors="ignore")
    X = X.select_dtypes(include=[np.number]).copy()
    # Không lọc theo giá trị trên toàn bộ dataset; kể cả cột toàn NaN
    # sẽ được đánh giá và loại trong training fold bởi FoldFeatureFilter.
    return X

class FoldFeatureFilter(BaseEstimator, TransformerMixin):
    """Lọc đặc trưng chỉ dựa trên training fold."""
    def __init__(self, missing_threshold=0.50, drop_range_columns=True):
        self.missing_threshold = missing_threshold
        self.drop_range_columns = drop_range_columns

    def fit(self, X, y=None):
        X = X.copy()
        cols = list(X.columns)
        if self.drop_range_columns:
            # Loại các biến min/max/range theo tên, áp dụng nhất quán cho mọi fold.
            cols = [
                c for c in cols
                if not re.search(r"(^|[_\s])(min|max|minimum|maximum|range)([_\s]|$)",
                                 str(c).lower())
            ]
        candidate = X.loc[:, cols]
        keep = []
        for col in candidate.columns:
            non_missing = candidate[col].dropna()
            if candidate[col].isna().mean() <= self.missing_threshold and non_missing.nunique() > 1:
                keep.append(col)
        self.keep_columns_ = keep
        if not self.keep_columns_:
            raise ValueError("Training fold không còn đặc trưng hợp lệ sau bước lọc.")
        return self

    def transform(self, X):
        missing = [c for c in self.keep_columns_ if c not in X.columns]
        if missing:
            raise ValueError(f"Thiếu cột ở transform: {missing[:5]}")
        return X.loc[:, self.keep_columns_]

class AdaptiveSelectKBest(BaseEstimator, TransformerMixin):
    """Chọn tối đa k đặc trưng; tự giảm k nếu fold có ít đặc trưng."""
    def __init__(self, k=10):
        self.k = k

    def fit(self, X, y):
        self.selector_ = SelectKBest(score_func=f_classif, k=min(self.k, X.shape[1]))
        self.selector_.fit(X, y)
        return self

    def transform(self, X):
        return self.selector_.transform(X)

def build_models():
    def preprocessing_steps():
        # Tạo estimator mới cho từng pipeline, tránh dùng chung trạng thái đã fit.
        return [
            ("feature_filter", FoldFeatureFilter(missing_threshold=0.50)),
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("select", AdaptiveSelectKBest(k=10))
        ]

    return {
        "Dummy baseline": Pipeline(preprocessing_steps() + [
            ("model", DummyClassifier(strategy="most_frequent"))
        ]),
        "Logistic Regression": Pipeline(preprocessing_steps() + [
            ("scale", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=3000, class_weight="balanced", C=0.1, random_state=42
            ))
        ]),
        "SVM (RBF)": Pipeline(preprocessing_steps() + [
            ("scale", StandardScaler()),
            ("model", SVC(kernel="rbf", class_weight="balanced", C=0.1))
        ]),
        "Random Forest": Pipeline(preprocessing_steps() + [
            ("model", RandomForestClassifier(
                n_estimators=300, max_depth=3, min_samples_leaf=2,
                class_weight="balanced", random_state=42
            ))
        ])
    }

## 7. Leave-One-Astronaut-Out evaluation

Mỗi lượt giữ lại một phi hành gia làm test, huấn luyện trên các phi hành gia còn lại. Các dự đoán test hợp lệ được gộp để tính metric. Cách này kiểm tra mô hình trên người chưa xuất hiện trong tập train, phù hợp hơn random split trong bối cảnh này.

Metric chính: **Balanced Accuracy và F1-score**. Accuracy được báo cáo bổ sung. Bảng fold ghi rõ số mẫu train/test và fold nào bị bỏ qua; nếu một fold test chỉ có một lớp thì không diễn giải Balanced Accuracy của fold đó như một đánh giá hai lớp đầy đủ.

In [ ]:
def evaluate_loao(name, df):
    df = df.copy().reset_index(drop=True)
    y = df["Phase"].map(TARGET_MAP)
    if y.isna().any() or y.nunique() != 2:
        raise ValueError(f"{name}: label thiếu hoặc không đủ hai lớp.")
    groups = df["Astronaut"].astype(str)
    X = make_feature_frame(df)
    if X.shape[1] == 0:
        raise ValueError(f"{name}: không còn cột đặc trưng số.")

    logo = LeaveOneGroupOut()
    models = build_models()
    records, fold_records = [], []

    for model_name, model in models.items():
        y_true_all, y_pred_all = [], []
        for train_idx, test_idx in logo.split(X, y, groups):
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            held_out = groups.iloc[test_idx].iloc[0]
            base_record = {
                "Dataset": name, "Model": model_name,
                "Held_Out_Astronaut": held_out,
                "N_Train": len(train_idx), "N_Test": len(test_idx),
                "Train_Classes": y_train.nunique(), "Test_Classes": y_test.nunique()
            }
            if y_train.nunique() < 2 or y_test.nunique() < 2:
                fold_records.append({
                    **base_record, "Fold_Valid": False,
                    "Reason": "Train hoặc test chỉ có một lớp"
                })
                continue

            fitted = clone(model)
            fitted.fit(X.iloc[train_idx], y_train)
            pred = fitted.predict(X.iloc[test_idx])
            y_true_all.extend(y_test.tolist())
            y_pred_all.extend(pred.tolist())

            fold_records.append({
                **base_record, "Fold_Valid": True, "Reason": "",
                "Accuracy": accuracy_score(y_test, pred),
                "Balanced_Accuracy": balanced_accuracy_score(y_test, pred),
                "F1": f1_score(y_test, pred, zero_division=0)
            })

        if y_true_all:
            records.append({
                "Dataset": name, "Model": model_name,
                "N_Evaluated": len(y_true_all),
                "Accuracy": accuracy_score(y_true_all, y_pred_all),
                "Balanced_Accuracy": balanced_accuracy_score(y_true_all, y_pred_all),
                "Precision": precision_score(y_true_all, y_pred_all, zero_division=0),
                "Recall_Postflight": recall_score(y_true_all, y_pred_all, zero_division=0),
                "Recall_Preflight": recall_score(y_true_all, y_pred_all, pos_label=0, zero_division=0),
                "F1": f1_score(y_true_all, y_pred_all, zero_division=0),
                "Confusion_Matrix": confusion_matrix(y_true_all, y_pred_all).tolist()
            })
        else:
            records.append({
                "Dataset": name, "Model": model_name, "N_Evaluated": 0,
                "Accuracy": np.nan, "Balanced_Accuracy": np.nan,
                "Precision": np.nan, "Recall_Postflight": np.nan,
                "Recall_Preflight": np.nan, "F1": np.nan,
                "Confusion_Matrix": None
            })

    return pd.DataFrame(records), pd.DataFrame(fold_records)

all_metrics, all_folds = [], []
for name, df in datasets.items():
    metrics, folds = evaluate_loao(name, df)
    all_metrics.append(metrics)
    all_folds.append(folds)

model_comparison = pd.concat(all_metrics, ignore_index=True)
fold_results = pd.concat(all_folds, ignore_index=True)

display(model_comparison.sort_values(["Dataset", "Balanced_Accuracy"], ascending=[True, False]))
display(fold_results)

## 8. Diễn giải khả năng tổng quát hóa

Không kết luận mô hình tốt chỉ dựa trên Accuracy. Cần:
- So sánh với Dummy baseline.
- Kiểm tra Balanced Accuracy và F1.
- Xem kết quả từng phi hành gia/fold; nếu metric dao động mạnh thì mô hình không ổn định.
- Nhớ rằng LOAO chỉ có 4 fold (4 phi hành gia), vì vậy độ bất định rất lớn.
- Metric cao trên tập này không chứng minh mô hình áp dụng tốt cho phi hành đoàn khác.

In [ ]:
for name in datasets:
    subset = model_comparison[model_comparison["Dataset"] == name].sort_values(
        "Balanced_Accuracy", ascending=False
    )
    print(f"\n{name}")
    if subset.empty:
        print("Không có kết quả hợp lệ.")
        continue
    best = subset.iloc[0]
    print(
        f"Mô hình có Balanced Accuracy cao nhất: {best['Model']} "
        f"(Balanced Accuracy={best['Balanced_Accuracy']:.3f}, "
        f"F1={best['F1']:.3f})."
    )
    dummy = subset[subset["Model"] == "Dummy baseline"]
    if not dummy.empty:
        delta = best["Balanced_Accuracy"] - dummy.iloc[0]["Balanced_Accuracy"]
        print(f"Chênh lệch Balanced Accuracy so với Dummy: {delta:+.3f}.")
    valid_folds = fold_results[
        (fold_results["Dataset"] == name) & (fold_results["Fold_Valid"] == True)
    ]
    for model_name, g in valid_folds.groupby("Model"):
        if "Balanced_Accuracy" in g:
            print(
                f"{model_name}: LOAO fold Balanced Accuracy "
                f"min={g['Balanced_Accuracy'].min():.3f}, "
                f"max={g['Balanced_Accuracy'].max():.3f}."
            )

## 9. Export kết quả

In [ ]:
OUT_DIR = BASE_DIR / "processed_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
label_validation.to_csv(OUT_DIR / "Label_Validation_Report.csv", index=False)
class_distribution.to_csv(OUT_DIR / "Label_Distribution_Report.csv", index=False)
mission_day_report.to_csv(OUT_DIR / "Mission_Day_Validation_Report.csv", index=False)
model_comparison.to_csv(OUT_DIR / "ML_Model_Comparison_LOAO.csv", index=False)
fold_results.to_csv(OUT_DIR / "ML_LOAO_Fold_Results.csv", index=False)

print("Đã xuất:")
for f in [
    "Label_Validation_Report.csv",
    "Label_Distribution_Report.csv",
    "Mission_Day_Validation_Report.csv",
    "ML_Model_Comparison_LOAO.csv",
    "ML_LOAO_Fold_Results.csv"
]:
    print("-", OUT_DIR / f)

## 11. Thử nghiệm bổ sung: tách nhãn "cấp tính" và "hồi phục"

Nhãn `Postflight` gốc gộp cả mẫu ngay sau khi về và mẫu ở thời điểm muộn hơn. Việc gộp có thể che khuất khác biệt giữa đáp ứng sớm và giai đoạn hồi phục. Đánh giá thêm ba kịch bản bằng LOAO:

| Kịch bản | CBC + Serum | Plasma Proteomics |
|---|---|---|
| **Nhãn gốc** | Pre (L-) vs mọi R+ | Pre vs mọi Post |
| **Cấp tính** | Pre vs R+1 | Pre vs Immediate_Postflight |
| **Hồi phục** | Pre vs R+82, R+194 | Pre vs Longterm_Postflight |

Số mẫu ở các kịch bản phụ ít hơn đáng kể. Luôn xem số lượng từng lớp và fold hợp lệ cùng với metric; không suy luận chắc chắn từ một metric cao khi chỉ có vài mẫu.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)  # F-test trên fold nhỏ có biến hằng

def select_scenario(name, df, scenario):
    """Trả về tập con mẫu theo kịch bản. Nhãn Phase giữ nguyên (Preflight/Postflight)."""
    if scenario == "Nhãn gốc":
        return df.copy()
    if name == "CBC + Serum":
        post = {"Cấp tính": ["R+1"], "Hồi phục": ["R+82", "R+194"]}[scenario]
        keep = df["Timepoint"].isin(["L-92", "L-44", "L-3"] + post)
    else:
        post = {"Cấp tính": "Immediate_Postflight", "Hồi phục": "Longterm_Postflight"}[scenario]
        keep = df["ImmediateVLongTerm"].isin(["Preflight", post])
    return df[keep].reset_index(drop=True)

SCENARIOS = ["Nhãn gốc", "Cấp tính", "Hồi phục"]
scen_metrics, scen_folds = [], []
for name, df in datasets.items():
    for scen in SCENARIOS:
        sub = select_scenario(name, df, scen)
        m, f = evaluate_loao(f"{name}", sub)
        m.insert(1, "Scenario", scen); f.insert(1, "Scenario", scen)
        m["N_Postflight"] = int((sub["Phase"] == "Postflight").sum())
        m["N_Preflight"] = int((sub["Phase"] == "Preflight").sum())
        scen_metrics.append(m); scen_folds.append(f)

scenario_comparison = pd.concat(scen_metrics, ignore_index=True)
scenario_folds = pd.concat(scen_folds, ignore_index=True)

show_cols = ["Dataset", "Scenario", "Model", "N_Preflight", "N_Postflight",
             "Balanced_Accuracy", "Recall_Preflight", "Recall_Postflight"]
display(scenario_comparison[show_cols].round(3))

## 12. Permutation test: kết quả có hơn ngẫu nhiên không?

Permutation test xáo nhãn **trong từng phi hành gia**, giữ nguyên số mẫu Pre/Post của mỗi người và chạy lại toàn bộ LOAO pipeline. Mỗi lượt hoán vị phải thực hiện lại feature filtering, imputation, feature selection và huấn luyện chỉ trên training fold.

`p = (1 + số lần hoán vị có Balanced Accuracy >= giá trị quan sát) / (1 + số lần hoán vị hợp lệ)`. Giá trị p là kiểm định danh nghĩa cho từng tổ hợp dataset/kịch bản/model. Nhiều kiểm định làm tăng nguy cơ dương tính giả; cần xem xét hiệu chỉnh đa kiểm định. Với số phi hành gia và mẫu nhỏ, kết quả chỉ mang tính thăm dò.

In [ ]:
def pooled_loao_ba(X, y, groups, model):
    """Balanced Accuracy gộp dự đoán out-of-fold của các fold LOAO hợp lệ."""
    yt, yp = [], []
    for tr, te in LeaveOneGroupOut().split(X, y, groups):
        if y.iloc[tr].nunique() < 2 or y.iloc[te].nunique() < 2:
            continue
        fitted = clone(model).fit(X.iloc[tr], y.iloc[tr])
        yt.extend(y.iloc[te].tolist())
        yp.extend(fitted.predict(X.iloc[te]).tolist())
    if not yt or len(np.unique(yt)) < 2:
        return np.nan
    return balanced_accuracy_score(yt, yp)

def permutation_test(df, model_name, n_perm, seed=42):
    df = df.reset_index(drop=True)
    y = df["Phase"].map(TARGET_MAP).reset_index(drop=True)
    groups = df["Astronaut"].astype(str).reset_index(drop=True)
    X = make_feature_frame(df).reset_index(drop=True)
    model = build_models()[model_name]
    if model_name == "Random Forest":
        model.set_params(model__n_estimators=100)

    observed = pooled_loao_ba(X, y, groups, model)
    rng = np.random.default_rng(seed)
    null = []
    for _ in range(n_perm):
        y_perm = y.copy()
        # Hoán vị trong từng phi hành gia để giữ nguyên số nhãn mỗi người.
        for g in groups.unique():
            idx = np.where(groups.to_numpy() == g)[0]
            y_perm.iloc[idx] = rng.permutation(y.iloc[idx].to_numpy())
        null.append(pooled_loao_ba(X, y_perm, groups, model))
    null = np.asarray(null, dtype=float)
    null = null[np.isfinite(null)]
    if not np.isfinite(observed) or len(null) == 0:
        return observed, null, np.nan
    p_value = (1 + np.sum(null >= observed - 1e-12)) / (1 + len(null))
    return observed, null, p_value

# Số hoán vị tăng lên để cải thiện độ phân giải p-value.
# Có thể mất thời gian vì mỗi lần phải chạy lại toàn bộ LOAO.
N_PERM = {"Logistic Regression": 499, "Random Forest": 199}
perm_rows = []
for name, df in datasets.items():
    for scen in ["Nhãn gốc", "Cấp tính"]:
        sub = select_scenario(name, df, scen)
        for model_name, n_perm in N_PERM.items():
            obs, null, p = permutation_test(sub, model_name, n_perm)
            perm_rows.append({
                "Dataset": name, "Scenario": scen, "Model": model_name,
                "Observed_BA": obs,
                "Null_mean": float(null.mean()) if len(null) else np.nan,
                "Null_95th_pct": float(np.percentile(null, 95)) if len(null) else np.nan,
                "N_perm_requested": n_perm, "N_perm_valid": len(null), "p_value": p
            })
permutation_results = pd.DataFrame(perm_rows)
display(permutation_results.round(3))

In [ ]:
scenario_comparison.to_csv(DATA_DIR / "ML_Scenario_Comparison_LOAO.csv", index=False)
scenario_folds.to_csv(DATA_DIR / "ML_Scenario_LOAO_Fold_Results.csv", index=False)
permutation_results.to_csv(DATA_DIR / "ML_Permutation_Test.csv", index=False)
print("Đã xuất: ML_Scenario_Comparison_LOAO.csv, ML_Scenario_LOAO_Fold_Results.csv, ML_Permutation_Test.csv")

## 10. Kết luận và giới hạn

- CBC + Serum: `Phase` được đối chiếu với tiền tố `L-` / `R+`; cần xác nhận quy ước này với metadata chính thức trước khi diễn giải sinh học.
- Plasma Proteomics: `Phase` được lấy từ metadata `PreVpost` và kiểm tra chéo với `Timepoint` nếu cột này có trong bảng.
- `Mission_Day` là số ngày tương đối được suy ra từ nhãn `L-` / `R+`, không phải ngày lịch. Mốc âm biểu thị trước phóng; mốc dương biểu thị sau khi trở về. Không xem khoảng cách số học giữa hai hệ mốc là thời gian thực liên tục.
- Dữ liệu chỉ gồm 4 phi hành gia và số mẫu nhỏ; số chiều đặc trưng lớn hơn nhiều số mẫu. Nguy cơ overfitting và độ bất định cao.
- LOAO có tối đa 4 fold. Một fold không có đủ hai lớp có thể bị bỏ qua; cần xem `ML_LOAO_Fold_Results.csv` và `ML_Scenario_LOAO_Fold_Results.csv`.
- Feature filtering, imputation, feature selection và scaling được fit trong từng training fold. Đây là điểm quan trọng để tránh leakage trong đánh giá.
- Permutation test dùng số hoán vị hữu hạn và nhiều tổ hợp kiểm định; p-value cần được xem là thăm dò, không phải bằng chứng xác nhận.
- Không kết luận mô hình tốt chỉ dựa trên Accuracy hoặc một fold. Nếu mô hình không vượt Dummy baseline ổn định, kết luận phù hợp là chưa có bằng chứng về khả năng tổng quát hóa tốt.
- Không chỉnh sửa nhãn, loại mẫu hoặc chọn metric chỉ nhằm làm kết quả đẹp hơn.

**Lưu ý khi báo cáo:** chỉ điền nhận xét định lượng sau khi chạy notebook trên đúng các file tích hợp. Không sử dụng lại các metric đã tính từ pipeline cũ vì pipeline cũ đã lọc đặc trưng trước khi chia fold.